# Qwen3-4B Multi-Agent Test Environment

## Setup
- Cell 1: pip install + 코드 전체 + 모델 로드 (최초 1회)
- Cell 2: 에이전트 생성 + 프롬프트 정의
- Cell 3+: 실험 (복사해서 쓰면 됨)

---

## API Reference

### Agent 생성
```python
agent = Agent("이름", "system prompt")
```

### 단일 추론
```python
# 기본 (thinking OFF, 256 tokens)
r = agent.say("질문")

# thinking ON + 토큰 늘리기
r = agent.say("질문", max_tokens=2048, thinking=True)

# 결과 접근
r["response"]   # 응답 텍스트
r["tokens"]     # 생성 토큰 수
r["time"]       # 소요 시간(초)
```

### 파라미터
| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `max_tokens` | 256 | 생성 최대 토큰. thinking=True면 2048 권장 |
| `thinking` | False | True: Qwen3 내부 추론(`<think>`) 활성화. 느리지만 정확도↑ |

### 프롬프트 변경
```python
agent.set_prompt("새 system prompt")
```

### A → B 통신
```python
result = send(a, b, "메시지")
result["sender"]["response"]    # A의 응답
result["receiver"]["response"]  # B의 응답
result["total_tokens"]          # 총 토큰
```

### A → B → C 체인
```python
result = chain([a, b, c], "메시지")
result["final"]          # 마지막 에이전트 응답
result["total_tokens"]   # 총 토큰
result["steps"]          # 각 단계별 결과 리스트
```

### ask() 헬퍼 (ABCD 문제용)
```python
r = ask(agent, MATH_PROMPT, "질문텍스트", max_tokens=512)
r["answer"]     # 추출된 답 (A/B/C/D 또는 N/A)
r["response"]   # 전체 응답
r["tokens"]     # 토큰 수
r["time"]       # 소요 시간
```

### 유틸리티
```python
extract_number("답은 42입니다")   # → 42.0
grade(got, expected)              # 채점 (10% 오차 허용)
```

In [ ]:
# ============================================================
# Cell 1: 모델 로드 (최초 1회)
# ============================================================
!pip install -q torch transformers accelerate datasets

import torch
import time
import re

# Global model/tokenizer
_model = None
_tokenizer = None
_device = None

def load_model(model_id: str = "Qwen/Qwen3-4B"):
    global _model, _tokenizer, _device
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if _device == "cuda" else torch.float32
    print(f"Device: {_device}")
    if _device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Loading {model_id}...")
    t0 = time.time()
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,
        device_map="auto" if _device == "cuda" else None,
    )
    if _device == "cpu":
        _model = _model.to(_device)
    params = sum(p.numel() for p in _model.parameters()) / 1e9
    print(f"Loaded in {time.time()-t0:.1f}s ({params:.1f}B params)")

class Agent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt
        self.history = []

    def say(self, message: str, max_tokens: int = 256, thinking: bool = False) -> dict:
        if _model is None:
            raise RuntimeError("load_model()을 먼저 실행하세요.")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": message},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=thinking
        )
        inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
        t0 = time.time()
        with torch.no_grad():
            output = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        elapsed = time.time() - t0
        response = _tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gen_tokens = output.shape[1] - inputs["input_ids"].shape[1]
        result = {"response": response, "tokens": gen_tokens, "time": round(elapsed, 1)}
        self.history.append({"input": message, **result})
        return result

    def set_prompt(self, new_prompt: str):
        self.system_prompt = new_prompt
        return self

    def __repr__(self):
        return f"Agent('{self.name}')"

def send(sender, receiver, message, max_tokens=256, verbose=True):
    s = sender.say(message, max_tokens=max_tokens)
    r = receiver.say(s["response"], max_tokens=max_tokens)
    if verbose:
        print(f"[{sender.name}] {s['tokens']}tok, {s['time']}s")
        print(f"  >> {s['response'][:200]}")
        print(f"[{receiver.name}] {r['tokens']}tok, {r['time']}s")
        print(f"  >> {r['response'][:200]}")
    return {"sender": s, "receiver": r, "tx_tokens": s["tokens"], "total_tokens": s["tokens"] + r["tokens"]}

def chain(agents, message, max_tokens=256, verbose=True):
    current = message
    results = []
    for agent in agents:
        r = agent.say(current, max_tokens=max_tokens)
        results.append({"agent": agent.name, **r})
        if verbose:
            print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
            print(f"  >> {r['response'][:200]}")
        current = r["response"]
    return {"steps": results, "final": results[-1]["response"], "total_tokens": sum(r["tokens"] for r in results)}

def extract_number(text):
    nums = re.findall(r'-?[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else -999

def grade(got, expected, tolerance=0.1):
    if expected == 0:
        return abs(got) < tolerance
    return abs(got - expected) / abs(expected) < tolerance

load_model("Qwen/Qwen3-4B")

In [ ]:
# ============================================================
# Cell 2: Planner-Solver Agents + MMLU 로더
# ============================================================
import random
from datasets import load_dataset
from itertools import permutations
import re as _re

# --- Agent roles ---
PLANNER_PROMPT = (
    "You are a Planner. You read questions and create step-by-step solving plans. "
    "You do NOT solve the problem yourself. "
    "Output a clear, numbered plan that another agent can follow to find the answer."
)

SOLVER_PROMPT = (
    "You are a Solver. You receive a plan from another agent and execute it step-by-step. "
    "You CANNOT see the original question. "
    "Follow the plan exactly, perform any calculations needed, and select the answer. "
    "Output ONLY a single letter: A, B, C, or D."
)

planner = Agent("Planner", PLANNER_PROMPT)
solver = Agent("Solver", SOLVER_PROMPT)

# --- MMLU 로드 ---
print("Loading MMLU...")
mmlu = load_dataset("cais/mmlu", "all", split="test")
ANSWER_MAP = {0: "A", 1: "B", 2: "C", 3: "D"}

# Domain mapping for Key Idea 4
DOMAIN_MAP = {
    "Math": [
        "elementary_mathematics", "high_school_mathematics",
        "high_school_statistics", "abstract_algebra", "college_mathematics",
    ],
    "Science": [
        "high_school_physics", "high_school_chemistry",
        "high_school_biology", "conceptual_physics", "anatomy",
    ],
    "CS": [
        "high_school_computer_science", "college_computer_science",
        "computer_security", "machine_learning", "electrical_engineering",
    ],
}

def format_mmlu(example):
    q = example["question"]
    choices = example["choices"]
    text = f"{q}\nA) {choices[0]}\nB) {choices[1]}\nC) {choices[2]}\nD) {choices[3]}"
    return {"text": text, "answer": ANSWER_MAP[example["answer"]], "subject": example["subject"]}

def sample_mmlu(n, subjects=None, seed=42):
    random.seed(seed)
    if subjects:
        pool = [ex for ex in mmlu if ex["subject"] in subjects]
    else:
        pool = list(mmlu)
    samples = random.sample(pool, min(n, len(pool)))
    return [format_mmlu(s) for s in samples]

def extract_choices(question_text):
    lines = question_text.split("\n")
    return "\n".join(l for l in lines if l.strip().startswith(("A)", "B)", "C)", "D)")))

def extract_answer(response):
    """Multi-strategy answer extraction."""
    text = response.strip()
    # Strategy 1: exact single letter
    if text.upper() in ("A", "B", "C", "D"):
        return text.upper()
    # Strategy 2: boxed
    m = _re.search(r'\\boxed\{([A-D])\}', text)
    if m: return m.group(1)
    # Strategy 3: "Answer: X" or "answer is X"
    m = _re.search(r'(?:answer|choice)[\s:is]*([A-D])\b', text, _re.IGNORECASE)
    if m: return m.group(1).upper()
    # Strategy 4: line-initial letter
    m = _re.search(r'^([A-D])[)\.]', text, _re.MULTILINE)
    if m: return m.group(1)
    # Strategy 5: first standalone A-D
    m = _re.search(r'\b([A-D])\b', text)
    if m: return m.group(1)
    return "N/A"

print(f"MMLU loaded: {len(mmlu)} questions")
print(f"Agents: {planner}, {solver}")

---
# Key Idea 1: Mutual Cognitive Context in Planner-Solver Communication
- **Planner (Tx)**: 문제를 보고 풀이 계획 작성 (직접 풀지 않음)
- **Solver (Rx)**: 문제를 못 봄, 계획만 받아서 실행 → 답 도출
- 고정 토큰 예산(48/96)에서 조건별 정확도 비교

| 조건 | Planner가 Solver를 앎 | Solver가 Planner를 앎 | 기대 |
|------|:-:|:-:|------|
| blind | X | X | 일반적 계획, Solver 불확실 |
| p_aware | O | X | Solver 맞춤 계획 (실행 가능하게) |
| s_aware | X | O | Solver가 계획을 신뢰 |
| mutual | O | O | 최적 계획 + 신뢰 실행 |

In [ ]:
# ============================================================
# Key Idea 1: Planner-Solver Mutual Cognition (MMLU 20문제)
# ============================================================

# --- Planner prompts ---
P_BLIND = (
    "You are a Planner. Read the question and create a step-by-step plan "
    "for another agent to solve it. They CANNOT see the question. "
    "Include the answer choices in your plan. Do NOT solve it yourself."
)
P_AWARE = (
    "You are a Planner. Read the question and create a step-by-step plan "
    "for a Solver agent to execute. The Solver is good at following clear instructions "
    "and performing calculations, but CANNOT see the question. "
    "Include the answer choices. Make each step concrete and executable. Do NOT solve it yourself."
)

# --- Solver prompts ---
S_BLIND = (
    "You are a Solver. Another agent created a plan for a question you cannot see. "
    "Execute the plan step-by-step and determine the answer. "
    "Output ONLY a single letter: A, B, C, or D."
)
S_AWARE = (
    "You are a Solver. A Planner agent analyzed a question and created a solving plan for you. "
    "The Planner's structure and logic are reliable — follow their steps. "
    "Execute the plan and determine the answer. "
    "Output ONLY a single letter: A, B, C, or D."
)

CONDITIONS = {
    "blind":   {"p_sys": P_BLIND, "s_sys": S_BLIND},
    "p_aware": {"p_sys": P_AWARE, "s_sys": S_BLIND},
    "s_aware": {"p_sys": P_BLIND, "s_sys": S_AWARE},
    "mutual":  {"p_sys": P_AWARE, "s_sys": S_AWARE},
}

TX_BUDGETS = [96, 192]

# Multi-step reasoning subjects only (no simple recall)
REASONING_SUBJECTS = [
    "high_school_physics",
    "high_school_chemistry",
    "high_school_mathematics",
    "high_school_statistics",
    "college_chemistry",
    "college_mathematics",
]
Q1 = sample_mmlu(20, subjects=REASONING_SUBJECTS, seed=42)
print(f"Sampled {len(Q1)} reasoning questions")
for q in Q1[:3]:
    print(f"  [{q['subject']}] {q['text'][:60]}... ans={q['answer']}")

def run_key1(questions):
    results = {b: {c: [] for c in CONDITIONS} for b in TX_BUDGETS}

    for qi, q in enumerate(questions):
        choices = extract_choices(q["text"])
        print(f"\n--- Q{qi+1}/{len(questions)}: [{q['subject']}] {q['text'][:70]}...")
        print(f"    Expected: {q['answer']}")

        for budget in TX_BUDGETS:
            for cond, cfg in CONDITIONS.items():
                # Planner: sees full question, creates plan within budget
                planner.set_prompt(cfg["p_sys"])
                p_out = planner.say(q["text"], max_tokens=budget)

                # Solver: sees ONLY the plan, must determine answer
                solver.set_prompt(cfg["s_sys"])
                s_msg = f"Plan:\n{p_out['response']}\n\nAnswer:"
                s_out = solver.say(s_msg, max_tokens=48)

                ans = extract_answer(s_out["response"])
                correct = ans == q["answer"]

                results[budget][cond].append({
                    "correct": correct,
                    "answer": ans,
                    "expected": q["answer"],
                    "p_tokens": p_out["tokens"],
                    "s_tokens": s_out["tokens"],
                })

        # Log @96tok (lower budget)
        b = TX_BUDGETS[0]
        row = "  ".join(
            f"{c}: {results[b][c][-1]['answer']}{'✓' if results[b][c][-1]['correct'] else '✗'}"
            for c in CONDITIONS
        )
        print(f"  @{b}tok: {row}")
        if qi == 0:
            print(f"    Plan(blind)>> {results[96]['blind'][-1].get('p_response', p_out['response'][:250])}")

    # Restore
    planner.set_prompt(PLANNER_PROMPT)
    solver.set_prompt(SOLVER_PROMPT)
    return results

print(f"\nRunning Key Idea 1 (Planner-Solver, fixed-budget)...")
k1_results = run_key1(Q1)

# Results table
print(f"\n{'Budget':<8} {'blind':<12} {'p_aware':<12} {'s_aware':<12} {'mutual':<12}")
print("-" * 56)
for budget in TX_BUDGETS:
    row = f"{budget}tok  "
    for cond in CONDITIONS:
        res = k1_results[budget][cond]
        n = len(res)
        acc = sum(r["correct"] for r in res) / n
        na = sum(1 for r in res if r["answer"] == "N/A")
        row += f"{acc*100:>4.0f}% ({na}NA)  "
    print(row)

# Interaction effect
print(f"\n--- Interaction Effect ---")
for cond in CONDITIONS:
    lo = TX_BUDGETS[0]
    hi = TX_BUDGETS[1]
    acc_lo = sum(r["correct"] for r in k1_results[lo][cond]) / len(k1_results[lo][cond])
    acc_hi = sum(r["correct"] for r in k1_results[hi][cond]) / len(k1_results[hi][cond])
    drop = acc_hi - acc_lo
    print(f"  {cond:<12} {hi}tok:{acc_hi*100:>4.0f}% → {lo}tok:{acc_lo*100:>4.0f}%  (drop:{drop*100:>+5.1f}%)")

---
# Key Idea 2: Stage-Wise CoT Switching (MMLU 10문제)
- **Tx**: `math_agent` 3단계 CoT (이해→추론→전달)
- **Rx**: `science_agent` (문제를 못 봄, 최종 메시지만 수신)
- Tx의 어느 단계에서 Rx 인지를 적용하느냐에 따라 통신 효율 변화

| Condition | Stage 1 (이해) | Stage 2 (추론) | Stage 3 (전달) | Rx |
|-----------|:-:|:-:|:-:|:-:|
| All General | General | General | General | Rx doesn't know Tx |
| All Aware | Aware | Aware | Aware | Rx knows Tx |
| Tx Switch | General | General | Aware | Rx doesn't know Tx |
| Both Switch | General | General | Aware | Rx knows Tx |

In [ ]:
# ============================================================
# Key Idea 2: MMLU 10문제 x 4조건 (Stage-Wise CoT Switching)
# Rx (science_agent) does NOT see the original question.
# Rx only receives Tx's Stage 3 output.
# ============================================================
Q2_math = sample_mmlu(5, domain="Math", seed=99)
Q2_sci = sample_mmlu(5, domain="Science", seed=100)
Q2 = Q2_math + Q2_sci
random.shuffle(Q2)
print(f"Sampled {len(Q2)} questions (5 Math + 5 Science)")

# Stage messages
def make_stages(question_text, aware):
    """Generate 3 CoT stage prompts. aware=True means audience-aware."""
    if aware:
        s1 = f"Read this question and identify key information relevant for a science expert.\nQuestion: {question_text}"
        s2 = "Based on your analysis, determine the answer. Focus on reasoning a science expert can verify.\n{prev}"
        s3 = "You must send a message to a science expert who CANNOT see the question. They need to determine the answer from your message alone. Be concise, use scientific terminology.\n{prev}"
    else:
        s1 = f"Read this question and identify key information.\nQuestion: {question_text}"
        s2 = "Based on your analysis, determine the answer.\n{prev}"
        s3 = "You must send a message to another agent who CANNOT see the question. They need to determine the answer from your message alone. Explain everything clearly.\n{prev}"
    return [s1, s2, s3]

cot_conditions = {
    "all_general":  {"stage_aware": [False, False, False], "rx_knows_tx": False},
    "all_aware":    {"stage_aware": [True, True, True],    "rx_knows_tx": True},
    "tx_switch":    {"stage_aware": [False, False, True],  "rx_knows_tx": False},
    "both_switch":  {"stage_aware": [False, False, True],  "rx_knows_tx": True},
}

def run_key2(questions):
    results = {c: [] for c in cot_conditions}

    for qi, q in enumerate(questions):
        print(f"\n--- Q{qi+1}/{len(questions)}: [{q['subject']}] {q['text'][:80]}...")
        print(f"    Expected: {q['answer']}")

        for cond, cfg in cot_conditions.items():
            # Tx CoT 3 stages
            stages_gen = make_stages(q["text"], False)
            stages_awr = make_stages(q["text"], True)
            
            math_agent.set_prompt(
                "You are a mathematics domain expert. "
                "Approach problems through mathematical reasoning."
            )

            prev = ""
            tx_total = 0
            for i, aware in enumerate(cfg["stage_aware"]):
                stage = stages_awr[i] if aware else stages_gen[i]
                msg = stage.format(prev=prev) if "{prev}" in stage else stage
                r = math_agent.say(msg, max_tokens=128)
                prev = r["response"]
                tx_total += r["tokens"]
            
            tx_msg_tokens = r["tokens"]  # Stage 3 output = transmitted message
            tx_message = prev

            # Rx: doesn't see question, only gets Stage 3 output
            if cfg["rx_knows_tx"]:
                science_agent.set_prompt(
                    "You are a natural science domain expert. "
                    "A mathematics expert analyzed a problem and sent you their conclusion. "
                    "Trust their mathematical reasoning. "
                    "Determine the answer from their message. "
                    "Put your answer inside \\boxed{X} where X is A, B, C, or D."
                )
            else:
                science_agent.set_prompt(
                    "You are a natural science domain expert. "
                    "Another agent analyzed a problem and sent you their message. "
                    "Determine the answer from their message. "
                    "Put your answer inside \\boxed{X} where X is A, B, C, or D."
                )

            rx_msg = f"Message from another agent:\n{tx_message}\n\nWhat is the correct answer? Put it inside \\boxed{{X}}."
            rx_out = science_agent.say(rx_msg, max_tokens=128)

            match = _re.search(r'\\boxed\{([A-D])\}', rx_out["response"])
            ans = match.group(1) if match else "N/A"
            correct = ans == q["answer"]

            print(f"  [{cond:12s}] CoT:{tx_total:3d}tok Msg:{tx_msg_tokens:3d}tok Rx:{rx_out['tokens']:3d}tok → {ans} {'✓' if correct else '✗'}")
            if cond == "all_general":
                print(f"    Tx S3>> {tx_message[:200]}")

            results[cond].append({
                "correct": correct,
                "tx_msg_tokens": tx_msg_tokens,
                "tx_cot_tokens": tx_total,
                "rx_tokens": rx_out["tokens"],
                "total_tokens": tx_total + rx_out["tokens"],
            })

    # Restore
    math_agent.set_prompt(MATH_PROMPT)
    science_agent.set_prompt(SCIENCE_PROMPT)
    return results

print("\nRunning Key Idea 2...")
k2_results = run_key2(Q2)

print(f"\n{'Condition':<15} {'Accuracy':<10} {'Tx Msg':<10} {'Tx CoT':<10} {'Rx':<8} {'Total':<8}")
print("-" * 60)
for cond, res in k2_results.items():
    acc = sum(r["correct"] for r in res) / len(res)
    avg_msg = sum(r["tx_msg_tokens"] for r in res) / len(res)
    avg_cot = sum(r["tx_cot_tokens"] for r in res) / len(res)
    avg_rx = sum(r["rx_tokens"] for r in res) / len(res)
    avg_tot = sum(r["total_tokens"] for r in res) / len(res)
    print(f"{cond:<15} {acc*100:>5.1f}%    {avg_msg:>6.1f}    {avg_cot:>6.1f}  {avg_rx:>6.1f}  {avg_tot:>6.1f}")

---
# Key Idea 4: Task Scheduling (MMLU 30문제, 도메인별 10개)
- **3 에이전트**: `Math`, `Science`, `CS` (고정, 각자 도메인 프롬프트)
- A→B→C 체인, 6순열 전부 테스트
- **첫 번째 에이전트만 문제를 보고**, 이후 에이전트는 이전 메시지만 수신
- 도메인 매칭이 중요: 물리 문제를 Science가 먼저 보면 유리

In [ ]:
# ============================================================
# Key Idea 4: MMLU 30문제 x 6순열 (Information Asymmetry Chain)
# Only the FIRST agent sees the original question.
# Each subsequent agent only receives the previous agent's message.
# ============================================================
Q4_math = sample_mmlu(10, domain="Math", seed=77)
Q4_sci  = sample_mmlu(10, domain="Science", seed=78)
Q4_cs   = sample_mmlu(10, domain="CS", seed=79)
Q4_all  = [(q, "Math") for q in Q4_math] + [(q, "Science") for q in Q4_sci] + [(q, "CS") for q in Q4_cs]
print(f"Sampled: Math={len(Q4_math)}, Science={len(Q4_sci)}, CS={len(Q4_cs)}, Total={len(Q4_all)}")

agent_names = ["Math", "Science", "CS"]
all_orders = list(permutations(agent_names))

DOMAIN_LABELS = {
    "Math": "mathematics",
    "Science": "natural science",
    "CS": "computer science",
}

def run_chain_mmlu(order, question_text):
    """3-agent chain. Only first agent sees the question. Others get previous message only."""
    prev = ""
    total_tokens = 0

    for i, name in enumerate(order):
        domain_label = DOMAIN_LABELS[name]

        if i == 0:
            # First agent: sees the question, must communicate to next
            AGENTS[name].set_prompt(
                f"You are ONLY a {domain_label} domain expert. "
                "You must analyze this problem and send a message to the next agent "
                "who CANNOT see the question. Include enough information for them to continue."
            )
            msg = f"Question: {question_text}\nAnalyze and send a message for the next agent."
        elif i == 1:
            # Middle agent: receives message, processes, forwards
            AGENTS[name].set_prompt(
                f"You are ONLY a {domain_label} domain expert. "
                "You received a message from the previous agent about a problem they analyzed. "
                "Add your expertise and forward a refined message to the next agent."
            )
            msg = f"Message from previous agent:\n{prev}\n\nRefine this analysis with your expertise and forward."
        else:
            # Last agent: receives message, gives final answer
            AGENTS[name].set_prompt(
                f"You are ONLY a {domain_label} domain expert. "
                "You received a message from previous agents about a problem. "
                "Based on their analysis, determine the final answer. "
                "Put your answer inside \\boxed{X} where X is A, B, C, or D."
            )
            msg = f"Message from previous agents:\n{prev}\n\nWhat is the correct answer? Put it inside \\boxed{{X}}."

        r = AGENTS[name].say(msg, max_tokens=128)
        prev = r["response"]
        total_tokens += r["tokens"]

    # Restore prompts
    for name in order:
        AGENTS[name].set_prompt(PROMPTS[name])

    match = _re.search(r'\\boxed\{([A-D])\}', prev)
    return match.group(1) if match else "N/A", total_tokens

# Run all permutations
k4_results = {}
for order in all_orders:
    order_key = "→".join(order)
    domain_scores = {"Math": [], "Science": [], "CS": []}
    total_tok = 0

    print(f"\n{'='*50}")
    print(f"  Order: {order_key}")
    print(f"{'='*50}")

    for q, domain in Q4_all:
        ans, tok = run_chain_mmlu(order, q["text"])
        correct = ans == q["answer"]
        domain_scores[domain].append(correct)
        total_tok += tok

    all_scores = [s for scores in domain_scores.values() for s in scores]
    k4_results[order_key] = {
        "accuracy": sum(all_scores) / len(all_scores),
        "by_domain": {d: sum(s)/len(s) for d, s in domain_scores.items()},
        "correct": sum(all_scores),
        "total": len(all_scores),
        "total_tokens": total_tok,
    }
    print(f"  Total: {sum(all_scores)}/{len(all_scores)} "
          f"(M:{sum(domain_scores['Math'])}/10 S:{sum(domain_scores['Science'])}/10 C:{sum(domain_scores['CS'])}/10)")

print("\nDone.")

In [ ]:
# ============================================================
# Key Idea 4: 결과 비교표
# ============================================================
sorted_k4 = sorted(k4_results.items(), key=lambda x: -x[1]["accuracy"])

print(f"{'Order':<20} {'Total':<10} {'Math':<8} {'Science':<8} {'CS':<8} {'Tokens'}")
print("-" * 65)
for order_key, r in sorted_k4:
    d = r["by_domain"]
    print(f"{order_key:<20} {r['accuracy']*100:>5.1f}%    "
          f"{d['Math']*100:>5.1f}%  {d['Science']*100:>5.1f}%  {d['CS']*100:>5.1f}%  {r['total_tokens']}")

print(f"\nBest:  {sorted_k4[0][0]} ({sorted_k4[0][1]['accuracy']*100:.1f}%)")
print(f"Worst: {sorted_k4[-1][0]} ({sorted_k4[-1][1]['accuracy']*100:.1f}%)")

# Domain-optimal: which order is best for each domain?
print("\nBest order per domain (first agent = encoder):")
for domain in ["Math", "Science", "CS"]:
    best = max(k4_results.items(), key=lambda x: x[1]["by_domain"][domain])
    print(f"  {domain} questions: {best[0]} ({best[1]['by_domain'][domain]*100:.0f}%)")